In [33]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.metrics import r2_score

In [34]:
X, y = make_regression(n_samples=200, n_features=1, noise=20, random_state=42)

In [35]:
X_b = np.c_[np.ones((200, 1)), X]

In [36]:
def compute_cost(X_b, y, theta):
    m = len(y)
    y_pred = X_b @ theta
    error = y_pred - y
    cost = (1 / (2 * m)) * np.sum(error ** 2)
    return cost

In [37]:
def gradient_descent(X_b, y, theta, alpha, n_iterations):
    m = len(y)
    cost_history = []
    for i in range(n_iterations):
        y_pred = X_b @ theta
        error = y_pred - y
        gradients = (1 / m) * X_b.T @ error
        theta = theta - alpha * gradients
        cost_history.append(compute_cost(X_b, y, theta))
    return theta, cost_history

In [ ]:
theta = np.zeros(2)
alpha = 0.01
n_iterations = 1000
theta_final, cost_history = gradient_descent(X_b, y, theta, alpha, n_iterations)
print('Intercept:', theta_final[0])
print('Slope:', theta_final[1])

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(range(n_iterations), cost_history, color='blue')
plt.title('Cost vs Iterations (Gradient Descent)')
plt.xlabel('Iterations')
plt.ylabel('Cost (MSE)')
plt.savefig('plots/01_cost_curve.png')
plt.close()
print('Plot 1 saved.')

In [ ]:
y_pred_scratch = X_b @ theta_final
ss_res = np.sum((y - y_pred_scratch) ** 2)
ss_tot = np.sum((y - np.mean(y)) ** 2)
r2_scratch = 1 - (ss_res / ss_tot)
sk_model = LinearRegression()
sk_model.fit(X, y)
r2_sklearn = sk_model.score(X, y)
print(f'Scratch  -> Intercept: {theta_final[0]:.4f}, Slope: {theta_final[1]:.4f}, R2: {r2_scratch:.4f}')
print(f'Sklearn  -> Intercept: {sk_model.intercept_:.4f}, Slope: {sk_model.coef_[0]:.4f}, R2: {r2_sklearn:.4f}')
X_line = np.array([[X.min()], [X.max()]])
X_line_b = np.c_[np.ones((2, 1)), X_line]

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(X, y, color='gray', alpha=0.5, label='Data')
plt.plot(X_line, X_line_b @ theta_final, color='blue', label=f'Scratch (R2={r2_scratch:.3f})')
plt.plot(X_line, sk_model.predict(X_line), color='red', linestyle='--', label=f'Sklearn (R2={r2_sklearn:.3f})')
plt.title('Best Fit Line - Scratch vs Sklearn')
plt.xlabel('X')
plt.ylabel('y')
plt.legend()
plt.savefig('plots/02_best_fit.png')
plt.close()
print('Plot 2 saved.')

In [ ]:
X2, y2 = make_regression(n_samples=100, n_features=1, noise=15, random_state=42)
X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.2, random_state=42)
degrees = [1, 4, 15]
colors = ['green', 'blue', 'red']
X_plot = np.linspace(X2.min(), X2.max(), 300).reshape(-1, 1)
plt.figure(figsize=(10, 6))
plt.scatter(X2, y2, color='gray', alpha=0.5, label='Data')
for deg, col in zip(degrees, colors):
    model = make_pipeline(PolynomialFeatures(deg), LinearRegression())
    model.fit(X2_train, y2_train)
    train_r2 = model.score(X2_train, y2_train)
    test_r2  = model.score(X2_test, y2_test)
    print(f'Degree {deg:2d} -> Train R2: {train_r2:.4f}  Test R2: {test_r2:.4f}')
    plt.plot(X_plot, model.predict(X_plot), color=col, label=f'Degree {deg} (Test R2={test_r2:.3f})')
plt.title('Polynomial Degree Comparison - Underfit / Fit / Overfit')
plt.xlabel('X')
plt.ylabel('y')
plt.legend()
plt.savefig('plots/03_poly_comparison.png')
plt.close()
print('Plot 3 saved.')

In [ ]:
poly = PolynomialFeatures(degree=15)
X2_train_poly = poly.fit_transform(X2_train)
X2_test_poly  = poly.transform(X2_test)
models = {
    'No Reg':      LinearRegression(),
    'Ridge(1.0)':  Ridge(alpha=1.0),
    'Lasso(0.01)': Lasso(alpha=0.01, max_iter=10000)
}
colors_reg = ['red', 'blue', 'green']
plt.figure(figsize=(10, 6))
for (name, model), col in zip(models.items(), colors_reg):
    model.fit(X2_train_poly, y2_train)
    plt.plot(model.coef_, label=name, color=col, marker='o', markersize=3)
plt.title('Regularization - Coefficient Values (Degree-15 Polynomial)')
plt.xlabel('Coefficient Index')
plt.ylabel('Coefficient Value')
plt.legend()
plt.savefig('plots/04_regularization.png')
plt.close()
print('Plot 4 saved.')

In [ ]:
print('\n========== SUMMARY ==========')
print(f'Linear Regression (Scratch) R2  : {r2_scratch:.4f}')
print(f'Linear Regression (Sklearn) R2  : {r2_sklearn:.4f}')